In [ ]:
!nvidia-smi
!pip install -q inference-gpu
!pip install -q git+https://github.com/roboflow/sports.git
!pip list | grep supervision

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["ROBOFLOW_API_KEY"] = secrets.get_secret("ROBOFLOW_API_KEY")
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

ROBOFLOW_API_KEY = secrets.get_secret("ROBOFLOW_API_KEY")
print("Keys loaded.")

In [ ]:
# === YOUR VIDEO ===
SOURCE_VIDEO_PATH = "/kaggle/input/datasets/ikhlasakhan/football-dataset2/Testing.mp4"
OUTPUT_VIDEO_PATH = "/kaggle/working/tracked_output.mp4"
OUTPUT_CSV_PATH   = "/kaggle/working/tracking.csv"

# === YOUR MODEL'S CLASS IDs (confirmed from Cell 7 output) ===
BALL_ID       = 1
GOALKEEPER_ID = 2
PLAYER_ID     = 3
REFEREE_ID    = 4
CONFIDENCE    = 0.3

print("Config set.")

In [ ]:
from inference import get_model

PLAYER_DETECTION_MODEL = get_model(
    model_id="football-vgiqa-3njno/2",
    api_key=ROBOFLOW_API_KEY
)
print("Player detection model loaded.")

In [ ]:
FIELD_DETECTION_MODEL = get_model(
    model_id="football-field-detection-f07vi/14",
    api_key=ROBOFLOW_API_KEY
)
print("Field detection model loaded.")

In [ ]:
import supervision as sv
import numpy as np
from sports.annotators.soccer import draw_pitch, draw_points_on_pitch
from sports.configs.soccer import SoccerPitchConfiguration
from sports.common.view import ViewTransformer

CONFIG = SoccerPitchConfiguration()

annotated_frame = draw_pitch(CONFIG)
sv.plot_image(annotated_frame)
print("Pitch config ready.")

In [ ]:
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONFIDENCE)[0]
detections = sv.Detections.from_inference(result)

print("=== YOUR MODEL CLASS IDs ===")
for i in range(len(detections)):
    print(f"class_id: {detections.class_id[i]}  |  name: {detections['class_name'][i]}  |  conf: {detections.confidence[i]:.2f}")

In [ ]:
from tqdm import tqdm
from sports.common.team import TeamClassifier
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {DEVICE}")

STRIDE = 30

crops = []
for frame in tqdm(
    sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=STRIDE),
    desc='Collecting crops'
):
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONFIDENCE)[0]
    detections = sv.Detections.from_inference(result)

    # only collect actual players for fitting
    player_only = detections[detections.class_id == PLAYER_ID]
    player_crops = [sv.crop_image(frame, xyxy) for xyxy in player_only.xyxy]
    crops += player_crops

print(f"Collected {len(crops)} player crops")

In [ ]:
team_classifier = TeamClassifier(device=DEVICE)
team_classifier.fit(crops)
print("Team classifier ready.")

In [ ]:
def resolve_goalkeepers_team_id(players, goalkeepers):
    goalkeepers_xy  = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy      = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)

    goalkeepers_team_id = []
    for gk_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(gk_xy - team_0_centroid)
        dist_1 = np.linalg.norm(gk_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

print("Goalkeeper function ready.")

In [ ]:
# Test on one frame before running full video
tracker_test = sv.ByteTrack()
tracker_test.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONFIDENCE)[0]
detections = sv.Detections.from_inference(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections = tracker_test.update_with_detections(detections=all_detections)

# split BEFORE changing class_ids
goalkeepers = all_detections[all_detections.class_id == GOALKEEPER_ID]
players     = all_detections[all_detections.class_id == PLAYER_ID]
referees    = all_detections[all_detections.class_id == REFEREE_ID]

if len(players) > 0:
    player_crops = [sv.crop_image(frame, xyxy) for xyxy in players.xyxy]
    players.class_id = team_classifier.predict(player_crops)

if len(goalkeepers) > 0 and len(players) > 0:
    goalkeepers.class_id = resolve_goalkeepers_team_id(players, goalkeepers)

if len(referees) > 0:
    referees.class_id = np.full(len(referees), -1, dtype=int)

all_detections = sv.Detections.merge([players, goalkeepers, referees])
all_detections.class_id = np.clip(all_detections.class_id, 0, 2).astype(int)

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25, height=21, outline_thickness=1
)

labels = [f"#{tid}" for tid in all_detections.tracker_id]

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(annotated_frame, all_detections)
annotated_frame = label_annotator.annotate(annotated_frame, all_detections, labels)
annotated_frame = triangle_annotator.annotate(annotated_frame, ball_detections)
sv.plot_image(annotated_frame)

print(f"Team 0 players: {(players.class_id == 0).sum()}")
print(f"Team 1 players: {(players.class_id == 1).sum()}")

In [ ]:
import cv2
import pandas as pd

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25, height=21, outline_thickness=1
)

tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=25
)
tracker.reset()

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
out = cv2.VideoWriter(
    OUTPUT_VIDEO_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    video_info.fps,
    (video_info.width, video_info.height)
)

tracking_records = []
frame_number = 0

for frame in tqdm(
    sv.get_video_frames_generator(SOURCE_VIDEO_PATH),
    total=video_info.total_frames,
    desc='Processing'
):
    # 1. Detect
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONFIDENCE)[0]
    detections = sv.Detections.from_inference(result)

    # 2. Separate ball
    ball_detections = detections[detections.class_id == BALL_ID]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    # 3. Track players
    player_detections = detections[detections.class_id != BALL_ID]
    player_detections = player_detections.with_nms(threshold=0.5, class_agnostic=True)
    player_detections = tracker.update_with_detections(detections=player_detections)

    # 4. Split by role BEFORE class_ids get changed
    goalkeepers = player_detections[player_detections.class_id == GOALKEEPER_ID]
    players     = player_detections[player_detections.class_id == PLAYER_ID]
    referees    = player_detections[player_detections.class_id == REFEREE_ID]

    # remember tracker_ids per role for CSV saving later
    gk_ids  = set(goalkeepers.tracker_id.tolist()) if goalkeepers.tracker_id is not None else set()
    ref_ids = set(referees.tracker_id.tolist()) if referees.tracker_id is not None else set()

    # 5. Classify teams
    if len(players) > 0:
        player_crops = [sv.crop_image(frame, xyxy) for xyxy in players.xyxy]
        players.class_id = team_classifier.predict(player_crops)

    if len(goalkeepers) > 0 and len(players) > 0:
        goalkeepers.class_id = resolve_goalkeepers_team_id(players, goalkeepers)

    if len(referees) > 0:
        referees.class_id = np.full(len(referees), -1, dtype=int)

    player_detections = sv.Detections.merge([players, goalkeepers, referees])

    # 6. Get pitch transformer
    has_transform = False
    try:
        result_field = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        key_points   = sv.KeyPoints.from_inference(result_field)
        kp_filter    = key_points.confidence[0] > 0.5
        frame_ref    = key_points.xy[0][kp_filter]
        pitch_ref    = np.array(CONFIG.vertices)[kp_filter]
        if len(frame_ref) >= 4:
            transformer   = ViewTransformer(source=frame_ref, target=pitch_ref)
            has_transform = True
    except:
        pass

    # 7. Save ball to CSV
    if len(ball_detections) > 0:
        for xyxy in ball_detections.xyxy:
            bx = float((xyxy[0] + xyxy[2]) / 2)
            by = float((xyxy[1] + xyxy[3]) / 2)
            px, py = (None, None)
            if has_transform:
                pt = transformer.transform_points(np.array([[bx, by]]))
                px, py = float(pt[0][0]), float(pt[0][1])
            tracking_records.append({
                'frame': frame_number,
                'player_id': -1,
                'x': bx, 'y': by,
                'pitch_x': px, 'pitch_y': py,
                'role': 'ball',
                'team_id': -1,
                'confidence': float(ball_detections.confidence[0])
            })

    # 8. Save players to CSV
    if player_detections.tracker_id is not None:
        positions      = player_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pitch_positions = transformer.transform_points(positions) if has_transform else [[None, None]] * len(positions)

        for i, tracker_id in enumerate(player_detections.tracker_id):
            tid = int(tracker_id)
            if tid in gk_ids:
                role = 'goalkeeper'
            elif tid in ref_ids:
                role = 'referee'
            else:
                role = 'player'

            tracking_records.append({
                'frame':      frame_number,
                'player_id':  tid,
                'x':          float(positions[i][0]),
                'y':          float(positions[i][1]),
                'pitch_x':    float(pitch_positions[i][0]) if has_transform else None,
                'pitch_y':    float(pitch_positions[i][1]) if has_transform else None,
                'role':       role,
                'team_id':    int(player_detections.class_id[i]),
                'confidence': float(player_detections.confidence[i])
            })

    # 9. Annotate and write frame
    if player_detections.class_id is not None:
        player_detections.class_id = np.where( player_detections.class_id == -1, 2, player_detections.class_id ).astype(int)

    labels = [f"#{tid}" for tid in player_detections.tracker_id] if player_detections.tracker_id is not None else []

    annotated_frame = frame.copy()
    annotated_frame = ellipse_annotator.annotate(annotated_frame, player_detections)
    annotated_frame = label_annotator.annotate(annotated_frame, player_detections, labels)
    annotated_frame = triangle_annotator.annotate(annotated_frame, ball_detections)
    out.write(annotated_frame)

    frame_number += 1

out.release()

# 10. Save CSV
df = pd.DataFrame(tracking_records)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f'\n=== DONE ===')
print(f'Frames processed: {frame_number}')
print(f'Total records: {len(df)}')
print(f'Roles: {df["role"].value_counts().to_dict()}')
print(f'Teams (players only): {df[df["role"].isin(["player","goalkeeper"])]["team_id"].value_counts().to_dict()}')
print(f'\nSample rows:')
print(df[df['role'] == 'player'][['frame','player_id','x','y','pitch_x','pitch_y','team_id']].head(8))

In [ ]:
df = pd.read_csv(OUTPUT_CSV_PATH)

print("=== SHAPE ===")
print(df.shape)
print("\n=== ROLE COUNTS ===")
print(df['role'].value_counts())
print("\n=== TEAM DISTRIBUTION (players + goalkeepers) ===")
print(df[df['role'].isin(['player','goalkeeper'])].groupby(['role','team_id']).size())
print("\n=== ID CONSISTENCY (Player #1 across frames) ===")
p = df[df['player_id'] == df[df['role']=='player']['player_id'].value_counts().index[0]]
print(p[['frame','player_id','x','y','team_id']].head(10))
print("\nIf team_id stays 0 or 1 and x/y changes gradually — everything is working.")